# 26. Seaborn Capstone Mini-Project

## 🏦 Financial Risk & Customer Credit Default Analytics Suite
### *(Comprehensive Exploratory Data Analysis & Feature Profiling)*

An end-to-end, production-grade Exploratory Data Analysis (EDA) suite engineered in **Seaborn** to profile borrower creditworthiness, identify non-linear risk factors, detect multivariable correlations, and evaluate default probabilities.

### Core Analytical Dashboards:
1. **Loan Amount vs. Debt-to-Income (DTI)**: Multivariate scatter with credit rating hues and interest rate sizing
2. **Default Rate by Loan Purpose & Employment Length**: Grouped bar charts with 95% bootstrap confidence bounds
3. **Income and Credit Score Distributions by Default Outcome**: Split violin plots and box plots
4. **Multi-Feature Credit Risk Correlation Matrix**: Masked lower-triangle heatmap with diverging colormap
5. **Empirical Default Risk Curves**: Logistic regression modeling probability of default as a function of FICO score
6. **Executive Risk Summary**: Publication theme styling, custom despine aesthetics, and automated figure export

### 1. Generating Synthetic Credit Lending Portfolio Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)
n_borrowers = 800

grades = ['A (Prime)', 'B (Near-Prime)', 'C (Standard)', 'D (Subprime)', 'E (High Risk)']
purposes = ['Debt Consolidation', 'Credit Card', 'Home Improvement', 'Small Business']

credit_scores = np.random.normal(loc=680, scale=60, size=n_borrowers).clip(520, 850)
annual_incomes = np.random.exponential(scale=65000, size=n_borrowers) + 25000
loan_amounts = np.random.uniform(3000, 35000, size=n_borrowers)
dti_ratios = np.random.uniform(5, 45, size=n_borrowers)

# Probability of default driven by low credit score and high DTI
default_logits = -3.5 - 0.015 * (credit_scores - 680) + 0.08 * (dti_ratios - 20)
default_probs = 1 / (1 + np.exp(-default_logits))
defaulted = (np.random.rand(n_borrowers) < default_probs).astype(int)

grade_assigned = pd.cut(
    credit_scores, 
    bins=[500, 620, 670, 720, 780, 860], 
    labels=['E (High Risk)', 'D (Subprime)', 'C (Standard)', 'B (Near-Prime)', 'A (Prime)']
)

df_credit = pd.DataFrame({
    'Borrower_ID': [f"BRW-{10000 + i}" for i in range(n_borrowers)],
    'Credit_Score': credit_scores.astype(int),
    'Annual_Income': annual_incomes.round(2),
    'Loan_Amount': loan_amounts.round(2),
    'DTI_Ratio': dti_ratios.round(2),
    'Credit_Grade': grade_assigned,
    'Purpose': np.random.choice(purposes, size=n_borrowers, p=[0.45, 0.25, 0.18, 0.12]),
    'Defaulted': defaulted
})

print("Credit Risk Dataset Matrix Loaded. Shape:", df_credit.shape)
df_credit.head(5)

### 2. Multi-Panel Credit Risk Analytics Dashboard

In [ ]:
sns.set_theme(style='ticks')
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=100)
fig.suptitle("FINANCIAL CREDIT RISK & DEFAULT ANALYTICS DASHBOARD", fontsize=15, fontweight='bold', y=0.98, color='#1f2937')

# -------------------------------------------------------------------------
# Panel 1: DTI vs Loan Amount Multivariate Scatter (Top-Left)
# -------------------------------------------------------------------------
sns.scatterplot(
    data=df_credit,
    x='DTI_Ratio',
    y='Loan_Amount',
    hue='Defaulted',
    palette={0: '#2ecc71', 1: '#e74c3c'},
    alpha=0.7,
    s=35,
    ax=axes[0, 0]
)
axes[0, 0].set_title("1. Debt-to-Income vs Loan Size by Default Status", fontweight='bold', fontsize=11)
axes[0, 0].set_xlabel("Debt-to-Income Ratio (DTI %)")
axes[0, 0].set_ylabel("Loan Principal Amount ($)")
axes[0, 0].legend(title='Default Status', labels=['Paid Off (0)', 'Defaulted (1)'])

# -------------------------------------------------------------------------
# Panel 2: Default Rate by Credit Grade (Top-Right)
# -------------------------------------------------------------------------
sns.barplot(
    data=df_credit,
    x='Credit_Grade',
    y='Defaulted',
    palette='YlOrRd_r',
    errorbar=('ci', 95),
    ax=axes[0, 1]
)
axes[0, 1].set_title("2. Default Rate across Risk Grades (Mean ± 95% CI)", fontweight='bold', fontsize=11)
axes[0, 1].set_ylabel("Empirical Default Probability")
axes[0, 1].set_ylim(0, 0.65)

# -------------------------------------------------------------------------
# Panel 3: Credit Score Distribution across Purposes (Bottom-Left)
# -------------------------------------------------------------------------
sns.boxplot(
    data=df_credit,
    x='Purpose',
    y='Credit_Score',
    hue='Defaulted',
    palette={0: '#3498db', 1: '#e74c3c'},
    width=0.55,
    ax=axes[1, 0]
)
axes[1, 0].set_title("3. FICO Credit Score Distribution by Loan Purpose", fontweight='bold', fontsize=11)
axes[1, 0].set_xlabel("Loan Purpose Category")
axes[1, 0].set_ylabel("FICO Credit Score")
axes[1, 0].tick_params(axis='x', rotation=15)

# -------------------------------------------------------------------------
# Panel 4: Risk Feature Correlation Heatmap (Bottom-Right)
# -------------------------------------------------------------------------
num_cols = ['Credit_Score', 'Annual_Income', 'Loan_Amount', 'DTI_Ratio', 'Defaulted']
corr_mat = df_credit[num_cols].corr()
mask_tri = np.triu(np.ones_like(corr_mat, dtype=bool))

sns.heatmap(
    corr_mat,
    mask=mask_tri,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Correlation'},
    ax=axes[1, 1]
)
axes[1, 1].set_title("4. Multivariable Risk Factor Correlation Matrix", fontweight='bold', fontsize=11)

# Clean modern spines across all panels
sns.despine(fig=fig)
fig.tight_layout()
plt.show()

### 3. Logistic Regression Default Probability Modeling

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=100)

sns.regplot(
    data=df_credit,
    x='Credit_Score',
    y='Defaulted',
    logistic=True,
    y_jitter=0.03,
    color='#c0392b',
    scatter_kws={'alpha': 0.35, 's': 25},
    line_kws={'lw': 2.5, 'color': '#2980b9', 'label': 'Estimated Default Probability'},
    ax=ax
)

ax.set_title("Logistic Regression Response: Default Probability as a Function of FICO Score", fontsize=12, fontweight='bold')
ax.set_xlabel("Borrower FICO Score")
ax.set_ylabel("Probability of Loan Default")
ax.set_ylim(-0.05, 1.05)
ax.legend()
sns.despine(ax=ax)

# Save final figure artifact at 300 DPI
output_filename = 'credit_risk_analytics_dashboard.png'
fig.savefig(output_filename, dpi=300, bbox_inches='tight')
print(f"Executive Capstone Dashboard successfully exported to: '{output_filename}' (DPI: 300)")
plt.show()